<a href="https://colab.research.google.com/github/ErnyBSB/ErnyBSB/blob/main/ProcessandoDocumentosComLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2025 Erny Bo-D

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

 https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.


---



| Author |
| --- |
| [Erny Bo-D](https://github.com/ErnyBSB) |



---
baseado no colab: *Document_processing_with_Gemini.ipynb*, disponível em Google Cloud.


# Processando Documentos com modelos LLM


## Visão Geral

Como em várias outras áreas, a área que arquivos e bibliotecas tem como objeto a ser **tratado, organizado, preservado e disponibilizado** aos usuários documentos, quer em texto, áudio, video, códigos, ou qualquer outro tipo de conteúdo e/ou formato (analógico, ou digital).


---


As ferramentas de IA, seus **modelos e técnicas de machine learning**, tornam possível uma série de tratamentos nesses documentos, como identificação de informações, resumos, extração de dados, tanto no documento em si, como também em seus acervos e coleções.


---


Exemplificamos aqui algumas dessas técnicas usando **recursos e ferramentas de modelos de IA**.




### Objectives

In this tutorial, you will learn how to use the Gemini API in Vertex AI with the Google Gen AI SDK for Python to process PDF documents.

You will complete the following tasks:

- Install the SDK
- Use the Gemini 2.0 Flash model to:
  - Extract structured entities from an unstructured document
  - Classify document types
  - Combine classification and entity extraction into a single workflow
  - Answer questions from documents
  - Summarize documents
  - Extract Table Data as HTML
  - Translate documents
  - Compare and contrast similar documents
  - Identify and extract relevant pages from a PDF

### Custos

Nas configurações deste Colab há custos (baixos) relativos ao uso do modelo Gemini no Google Cloud, o qual deve ser previamente configurado.


## Configurações básicas


### Instalação **Google Gen AI SDK for Python**


In [ ]:
%pip install --upgrade --quiet google-genai pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.7/724.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 15.4 MB/s eta 0:00:00


### Autenticação no ambiente Colab do Google Cloud

Execute a célula seguinte para se autenticar. Em outros ambientes de testes, pode não ser necessário esta autenticação.


In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
  # Authenticate user to Google Cloud
  from google.colab import auth

  auth.authenticate_user()

### Escolhendo um projeto no Google Cloud

A partir do ambiente Google Cloud (Console) é preciso criar um projeto a ser vinculado aos testes neste Colab.

In [ ]:
import os

from google import genai

PROJECT_ID = "regal-spark-487121-g4"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
  PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

### Importar bibliotecas e setar variáveis


In [ ]:
from datetime import date
from enum import Enum
import json

from google.genai.types import GenerateContentConfig, Part
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
import pypdf

PDF_MIME_TYPE = "application/pdf"
JSON_MIME_TYPE = "application/json"
ENUM_MIME_TYPE = "text/x.enum"

### Especificando o modelo LLM a ser utilizado


Utilizaremos o modelo Gemini 2.0 Flash (`gemini-2.0-flash`).

Verifique mais detalhes [Gemini models on Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models#gemini-models).

In [ ]:
MODEL_ID = "gemini-2.0-flash"  # @param {type: "string"}

## Entity Extraction

[Named Entity Extraction](https://en.wikipedia.org/wiki/Named-entity_recognition) is a technique of Natural Language Processing to identify specific fields and values from unstructured text. For example, you can find key-value pairs from a filled out form, or get all of the important data from an invoice categorized by the type.

### Extract entities from an invoice

In this example, you will use a sample invoice and get all of the information in a structured format.

This is the prompt to be sent to Gemini along with the PDF document. Feel free to edit this for your specific use case.

In [ ]:
entity_extraction_system_instruction = """You are a document entity extraction specialist. Given a document, your task is to extract the text value of the entities provided in the schema.
- The values must only include text found in the document
- Do not normalize any entity values.
"""

We will use [Controlled generation](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/control-generated-output) to tell the model which fields need to be extracted.

The response schema is specified in the `response_schema` parameter in `config`, and the model output will strictly follow that schema.

You can provide the schemas as [Pydantic](https://docs.pydantic.dev/) models or a [JSON](https://www.json.org/json-en.html) string and the model will respond as JSON or an [Enum](https://docs.python.org/3/library/enum.html) depending on the value set in `response_mime_type`.

In [ ]:
class Address(BaseModel):
  street: str | None = Field(None, example="123 Main St")
  city: str | None = Field(None, example="Springfield")
  state: str | None = Field(None, example="IL")
  postal_code: str | None = Field(None, example="62704")
  country: str | None = Field(None, example="USA")


class LineItem(BaseModel):
  amount: float = Field(..., example=100.00)
  description: str | None = Field(None, example="Laptop")
  product_code: str | None = Field(None, example="LPT-001")
  quantity: int = Field(..., example=2)
  unit: str | None = Field(None, example="pcs")
  unit_price: float = Field(..., example=50.00)


class VAT(BaseModel):
  amount: float = Field(..., example=20.00)
  category_code: str | None = Field(None, example="A")
  tax_amount: float | None = Field(None, example=5.00)
  tax_rate: float | None = Field(
      None, example=10.0
  )  # Percentage as a float (e.g., 10 for 10%)
  total_amount: float = Field(..., example=200.00)


class Party(BaseModel):
  name: str = Field(..., example="Google")
  street: str | None = Field(None, example="456 Business Rd")
  city: str | None = Field(None, example="Metropolis")
  state: str | None = Field(None, example="NY")
  postal_code: str | None = Field(None, example="10001")
  country: str | None = Field(None, example="USA")
  email: str | None = Field(None, example="contact@google.com")
  phone: str | None = Field(None, example="+1-555-1234")
  website: str | None = Field(None, example="https://google.com")
  tax_id: str | None = Field(None, example="123456789")
  registration: str | None = Field(None, example="Reg-98765")
  iban: str | None = Field(None, example="US1234567890123456789")
  payment_ref: str | None = Field(None, example="INV-2024-001")


class Invoice(BaseModel):
  invoice_id: str = Field(..., example="INV-2024-001")
  invoice_date: str = Field(..., example="2024-02-03")
  supplier: Party
  receiver: Party
  line_items: list[LineItem]
  vat: list[VAT]

class Telegram(BaseModel):
  telegramFrom: str = Field(..., example="Bahia")
  telegramNum: str = Field(..., example="545")
  telegramHour: str = Field(..., example="12,00")
  telegramMessage: str = Field(..., example="")

class CartaDatilografada(BaseModel):
  carta_datilogra_date: str = Field(..., example= "São Paulo 25 de março de 1970")
  carta_datilogra_content: str = Field(..., example= "Caros amigos espero que estejam bem quando receberem esta carta...")

/tmp/ipython-input-3755437115.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  street: str | None = Field(None, example="123 Main St")
/tmp/ipython-input-3755437115.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  city: str | None = Field(None, example="Springfield")
/tmp/ipython-input-3755437115.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be r

Para o exemplo aqui criado, iremos carregar um documento de arquivo, digitalizado e disponível no formato PDF.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# pdf_doc = "/content/drive/MyDrive/DocsToAnalyse/BR-DFCD-BERTHALUTZ-BL1-16.pdf"
pdf_doc = "/content/drive/MyDrive/DocsToAnalyse/BR-DFCD-BERTHALUTZ-BL1-07.pdf"

In [ ]:
# Load file bytes
with open(pdf_doc, "rb") as f:
  file_bytes = f.read()

# Send to Gemini API
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "The following document is a telegram.",
        Part.from_bytes(data=file_bytes, mime_type=PDF_MIME_TYPE),
    ],
    config=GenerateContentConfig(
        system_instruction=entity_extraction_system_instruction,
        temperature=0,
        response_schema=Telegram,
        response_mime_type=JSON_MIME_TYPE,
    ),
)

O campo `response.parsed` contém os dados extraídos.

In [ ]:
telegram_data = response.parsed
print("\n-------Extracted Entities--------")
print(telegram_data)


-------Extracted Entities--------
telegramFrom='Bahia' telegramNum='266' telegramHour='20,30' telegramMessage='Federação Bahiana Progresso Feminino " Sotiesta preclaro espirito\njustiça Vocencia nomeação Fertha Lutz, representante movimento femi-\nnista nacional commissão ante projecto constituição. Nenhuma brasi-\nleira desempenhará mais dinamente honroso encargo, nem explanará\nmelhor causa feminina a que se vem devotando heroicamente largos\ndez annos. Paudações.\nDirectori\ndith Gama Abreu, Marietta Passo Cunha,\nLili Tosta Laurentina Pugas Tavares, Anisia Seabra,\nCeleste Cerdeira, Alice Kelsch Aguiar, Maria Luiza\nCarvalho, Lily Klein Schmidt e Beatrz Caria.'


Também pode-se gerar os dados em um dicionário python

In [ ]:
json_object = json.loads(response.text)
print(json_object)

{'telegramFrom': 'Bahia', 'telegramNum': '266', 'telegramHour': '20,30', 'telegramMessage': 'Federação Bahiana Progresso Feminino " Sotiesta preclaro espirito\njustiça Vocencia nomeação Fertha Lutz, representante movimento femi-\nnista nacional commissão ante projecto constituição. Nenhuma brasi-\nleira desempenhará mais dinamente honroso encargo, nem explanará\nmelhor causa feminina a que se vem devotando heroicamente largos\ndez annos. Paudações.\nDirectori\ndith Gama Abreu, Marietta Passo Cunha,\nLili Tosta Laurentina Pugas Tavares, Anisia Seabra,\nCeleste Cerdeira, Alice Kelsch Aguiar, Maria Luiza\nCarvalho, Lily Klein Schmidt e Beatrz Caria.'}


Verifique se os dados foram extraídos.

## Document Classification

Document classification is the process for identifying the type of document. For example, invoice, W-2, receipt, etc.

In this example, you will use a [sample tax form (W-9)](https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/w9.pdf) and get the specific type of document from a specified `Enum`.

In [ ]:
classification_prompt = """You are a document classification specialist. Given a document, your task is to find which category the document belongs to from the document categories provided in the schema."""


class DocumentCategory(Enum):
  TELEGRAM = "Telegrama" # use for paper copies of old telegram documents
  CARTA_DATILOGRAFA = "Carta datilografada" # use for machine typed letters
  CARTA_MANUSCRITA = "Carta manuscrita" # use for manuscript letters
  RECORTE_JORNAL = "Recorte de jornal" # use for cropped news papers parts


In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "Classify the following document.",
        Part.from_uri(
            file_uri="https://arquivohistorico.camara.leg.br/atom/BERTHALUTZ/BR-DFCD-BERTHALUTZ-BL1-26.pdf",
            mime_type=PDF_MIME_TYPE,
        ),
    ],
    config=GenerateContentConfig(
        system_instruction=classification_prompt,
        temperature=0,
        response_schema=DocumentCategory,
        response_mime_type=ENUM_MIME_TYPE,
    ),
)

In [ ]:
print("\n-------Document Classification--------")
print(response.text)
print(response.parsed)


-------Document Classification--------
Carta datilografada
DocumentCategory.CARTA_DATILOGRAFA


O documento foi corretamente classificado no script.


---


# Como usar os exemplos no WorkShop:

O exemplo de extração de informações no Telegrama e identificação do tipo de documento servem como ponto de partida para os seguintes pontos:

  * 1 - notar que não estamos usando uma ferramenta como o chatGPT, ou gemini
  * 2 - mesmo assim estamos acessando (diretamente) um modelo LLM
  * 3 - quais são as vantagens de fazer isso?
  * 3.1 melhor granularidade e customização de parâmetros
  * 3.2 capacidade de executar o script com centenas, ou milhares de documentos
  * 3.3 maior precisão na classificação, que q defino as possíveis alternativas
  * 3.4 posso criar customização como classificar e depois extrair (abaixo)
  * 4 SIM... os documentos precisam estar digitalizados


---




### Chaining Classification and Extraction

These techniques can also be chained together to extract any number of document types.

For example, if you have multiple types of documents to process, you can send each document to Gemini with a classification prompt, then based on that output, you can write logic to decide which extraction prompt to use.

These are the sample documents:

- [US Driver License](https://storage.googleapis.com/cloud-samples-data/documentai/SampleDocuments/US_DRIVER_LICENSE_PROCESSOR/dl3.pdf)
- [Invoice](https://storage.googleapis.com/cloud-samples-data/documentai/SampleDocuments/INVOICE_PROCESSOR/google_invoice.pdf)
- [Form W-2](https://storage.googleapis.com/cloud-samples-data/documentai/SampleDocuments/FORM_W2_PROCESSOR/2020FormW-2.pdf)

In [ ]:
class W2Form(BaseModel):
  control_number: str | None = Field(None)
  ein: str = Field(...)

  employee_first_name: str = Field(...)
  employee_last_name: str = Field(...)
  employee_address_street: str = Field(...)
  employee_address_city: str = Field(...)
  employee_address_state: str = Field(...)
  employee_address_zip: str = Field(...)

  employer_name: str = Field(...)
  employer_address_street: str = Field(...)
  employer_address_city: str = Field(...)
  employer_address_state: str = Field(...)
  employer_address_zip: str = Field(...)
  employer_state_id_number: str | None = Field(None)

  wages_tips_other_compensation: float = Field(...)
  federal_income_tax_withheld: float = Field(...)
  social_security_wages: float = Field(...)
  social_security_tax_withheld: float = Field(...)
  medicare_wages_and_tips: float = Field(...)
  medicare_tax_withheld: float = Field(...)

  state: str | None = Field(None)
  state_wages_tips_etc: float | None = Field(None)
  state_income_tax: float | None = Field(None)

  box_12_code: str | None = Field(None)
  box_12_value: str | None = Field(None)

  form_year: int = Field(...)


class DriversLicense(BaseModel):
  address: str = Field(
      ..., title="Address", description="The address of the individual."
  )
  date_of_birth: date = Field(
      ..., title="Date of Birth", description="The birthdate of the individual."
  )
  document_id: str = Field(
      ...,
      title="Document ID",
      description="The unique document ID for the driver's license.",
  )
  expiration_date: date = Field(
      ...,
      title="Expiration Date",
      description="The expiration date of the driver's license.",
  )
  family_name: str = Field(
      ...,
      title="Family Name",
      description="The family name (last name) of the individual.",
  )
  given_names: str = Field(
      ...,
      title="Given Names",
      description="The given names (first and middle names) of the individual.",
  )
  issue_date: date = Field(
      ...,
      title="Issue Date",
      description="The issue date of the driver's license.",
  )


# Map classification types to schemas
classification_to_schema = {
    DocumentCategory.INVOICE: Invoice,
    DocumentCategory.W2: W2Form,
    DocumentCategory.DRIVER_LICENSE: DriversLicense,
}

In [ ]:
gcs_uris = [
    "gs://cloud-samples-data/documentai/SampleDocuments/US_DRIVER_LICENSE_PROCESSOR/dl3.pdf",
    "gs://cloud-samples-data/documentai/SampleDocuments/INVOICE_PROCESSOR/google_invoice.pdf",
    "gs://cloud-samples-data/documentai/SampleDocuments/FORM_W2_PROCESSOR/2020FormW-2.pdf",
]

for gcs_uri in gcs_uris:
  print(f"\nFile: {gcs_uri}\n")

  # Send to Gemini with Classification Prompt
  classification_response = client.models.generate_content(
      model=MODEL_ID,
      contents=[
          "Classify the following document.",
          Part.from_uri(file_uri=gcs_uri, mime_type=PDF_MIME_TYPE),
      ],
      config=GenerateContentConfig(
          system_instruction=classification_prompt,
          temperature=0,
          response_schema=DocumentCategory,
          response_mime_type=ENUM_MIME_TYPE,
      ),
  )

  print(f"Document Classification: {classification_response.text}")

  # Get Extraction schema based on Classification
  extraction_schema = classification_to_schema.get(
      classification_response.parsed
  )

  if not extraction_schema:
    print(
        f"Document does not belong to a specified class. Skipping extraction."
    )
    continue

  # Send to Gemini with Extraction Prompt
  extraction_response = client.models.generate_content(
      model=MODEL_ID,
      contents=[
          (
              "Extract the entities from the following"
              f" {classification_response.text} document."
          ),
          Part.from_uri(file_uri=gcs_uri, mime_type=PDF_MIME_TYPE),
      ],
      config=GenerateContentConfig(
          system_instruction=classification_prompt,
          temperature=0,
          response_schema=extraction_schema,
          response_mime_type=JSON_MIME_TYPE,
      ),
  )

  print("\n-------Extracted Entities--------")
  print(extraction_response.parsed)


File: gs://cloud-samples-data/documentai/SampleDocuments/US_DRIVER_LICENSE_PROCESSOR/dl3.pdf

Document Classification: driver_license

-------Extracted Entities--------
address='123 MAIN STREETHELENA, MT 59601' date_of_birth=datetime.date(804, 10, 8) document_id='0812319684104' expiration_date=datetime.date(804, 10, 8) family_name='LYNN' given_names='BRENDA' issue_date=datetime.date(215, 10, 2)

File: gs://cloud-samples-data/documentai/SampleDocuments/INVOICE_PROCESSOR/google_invoice.pdf

Document Classification: invoice

-------Extracted Entities--------
invoice_id='23413561D' invoice_date='Sep 24, 2019' supplier=Party(name='Google', street=None, city=None, state=None, postal_code=None, country=None, email=None, phone=None, website=None, tax_id=None, registration=None, iban=None, payment_ref=None) receiver=Party(name='Jane Smith', street='1600 Amphitheatre Pkway', city='Mountain View', state='CA', postal_code='94043', country=None, email=None, phone=None, website=None, tax_id=None, r

## Document Question Answering

Gemini can be used to answer questions about a document.

This example answers a question about the Transformer model paper ["Attention is all you need"](https://arxiv.org/pdf/1706.03762), we will be loading the PDF file directly from the source on [arXiv](https://arxiv.org)

In [ ]:
qa_system_instruction = (
    "You are a question answering specialist. Given a question and a context,"
    " your task is to provide the answer to the question based on the context"
    " provided. Give the answer first, followed by an explanation."
)

In [ ]:
# Interrogando sobre o documento
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "Qual é o assunto principal do documento?",
        Part.from_uri(
            file_uri=(
                "https://arquivohistorico.camara.leg.br/atom/BERTHALUTZ/BR-DFCD-BERTHALUTZ-BL1-26.pdf"
            ),
            mime_type=PDF_MIME_TYPE,
        ),
    ],
    config=GenerateContentConfig(
        system_instruction=qa_system_instruction,
        temperature=0,
        response_mime_type="text/plain",
    ),
)

print(f"Answer: {response.text}")

Answer: O assunto principal do documento é a defesa dos direitos das mulheres na futura Constituição do Brasil, com foco na igualdade jurídica, econômica e política, e na exclusão do serviço militar obrigatório para as mulheres.

O documento é uma carta da Federação Brasileira pelo Progresso Feminino (FBPF) aos membros da Assembleia Nacional Constituinte em fevereiro de 1934. A carta expressa preocupações e reivindicações das mulheres brasileiras em relação à nova Constituição que estava sendo elaborada. A FBPF argumenta pela igualdade de direitos entre homens e mulheres, incluindo o direito ao voto e a ocupação de cargos públicos, e se opõe à inclusão das mulheres no serviço militar obrigatório.


## Document Summarization

Gemini can also be used to summarize or paraphrase a document's contents. Your prompt can specify how detailed the summary should be or specific formatting, such as bullet points or paragraphs.

In [ ]:
summarization_system_instruction = """You are a professional document summarization specialist. Given a document, your task is to provide a detailed summary of the content of the document.

If it includes images, provide descriptions of the images.
If it includes tables, extract all elements of the tables.
If it includes graphs, explain the findings in the graphs.
Do not include any numbers that are not mentioned in the document.
"""

In [ ]:
# Send Summarization Prompt to Gemini
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        "Summarize the following document.",
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/pdf/fdic_board_meeting.pdf",
            mime_type=PDF_MIME_TYPE,
        ),
    ],
    config=GenerateContentConfig(
        system_instruction=summarization_system_instruction,
        temperature=0,
        response_mime_type="text/plain",
    ),
)

display(Markdown(f"### Document Summary"))
display(Markdown(response.text))

### Document Summary

The document is a statement by FDIC Chairman Jelena McWilliams on the Notice of Proposed Rulemaking on Revisions to the Community Reinvestment Act Regulations at the FDIC Board Meeting on December 12, 2019. The statement discusses modernizing the regulations implementing the Community Reinvestment Act (CRA) to promote greater investments in the communities that need them most. The last major revisions to the regulations implementing the CRA were made in 1995.

The principal goal in undertaking these reforms is to increase LMI lending. The proposal seeks to achieve that goal in several ways, including:
*   Encouraging banks to make long-term commitments in LMI communities by providing greater credit for retail loans retained on-balance sheet.
*   Increasing to $2 million the size of qualifying loans to small businesses and small farms to encourage economic development and job creation and help family farms.
*   Providing CRA credit for retail and community development activities in Indian Country.
*   Expanding the activities that qualify for CRA credit to include capital investments and loan participations undertaken by a bank in cooperation with Community Development Financial Institutions (CDFIs), regardless of where the CDFI is located.

The proposal would clarify the activities that qualify for CRA credit in two key ways. First, it would require the FDIC and OCC to publish periodically a list of illustrative examples of qualifying activities, and establish a process for stakeholders to seek agency determination of a qualifying activity. Second, it would establish new performance standards to assess: (1) the distribution of qualifying retail loan originations to LMI individuals, and to small farms and small businesses in an assessment area; and (2) the quantified value of the bank's qualifying activities relative to its assessment area and bank-level retail deposits.

The proposal would also recognize the evolution of the banking system – including the emergence of digital banks – by requiring banks to add assessment areas where they have significant concentrations of retail domestic deposits.

The proposal would leave intact existing provisions for banks to delineate assessment areas where they have their main office, branches, and deposit-taking facilities, and to include in those areas the surrounding geographies where banks originate or purchase a substantial portion of their loans.

The proposal would allow small banks (i.e., those with $500 million or less in total assets) to select to be evaluated under the current rules or to opt in to the new performance standards.

## Table parsing from documents

Gemini can parse contents of a table and return it in a structured format, such as HTML or markdown.

In [ ]:
table_extraction_prompt = (
    """What is the HTML code of the table in this document?"""
)

In [ ]:
# Send Table Extraction Prompt to Gemini
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        table_extraction_prompt,
        Part.from_uri(
            file_uri=(
                "gs://cloud-samples-data/generative-ai/pdf/salary_table.pdf"
            ),
            mime_type=PDF_MIME_TYPE,
        ),
    ],
    config=GenerateContentConfig(temperature=0),
)

display(Markdown(response.text))

```html
<table>
  <thead>
    <tr>
      <th>Grade</th>
      <th>Step 1</th>
      <th>Step 2</th>
      <th>Step 3</th>
      <th>Step 4</th>
      <th>Step 5</th>
      <th>Step 6</th>
      <th>Step 7</th>
      <th>Step 8</th>
      <th>Step 9</th>
      <th>Step 10</th>
      <th>WITHIN GRADE AMOUNTS</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
      <td>$ 20,999</td>
      <td>$ 21,704</td>
      <td>$ 22,401</td>
      <td>$ 23,097</td>
      <td>$ 23,794</td>
      <td>$ 24,202</td>
      <td>$ 24,893</td>
      <td>$ 25,589</td>
      <td>$ 25,617</td>
      <td>$ 26,273</td>
      <td>VARIES</td>
    </tr>
    <tr>
      <td>2</td>
      <td>23,612</td>
      <td>24,174</td>
      <td>24,956</td>
      <td>25,617</td>
      <td>25,906</td>
      <td>26,668</td>
      <td>27,430</td>
      <td>28,192</td>
      <td>28,954</td>
      <td>29,716</td>
      <td>VARIES</td>
    </tr>
    <tr>
      <td>3</td>
      <td>25,764</td>
      <td>26,623</td>
      <td>27,482</td>
      <td>28,341</td>
      <td>29,200</td>
      <td>30,059</td>
      <td>30,918</td>
      <td>31,777</td>
      <td>32,636</td>
      <td>33,495</td>
      <td>859</td>
    </tr>
    <tr>
      <td>4</td>
      <td>28,921</td>
      <td>29,885</td>
      <td>30,849</td>
      <td>31,813</td>
      <td>32,777</td>
      <td>33,741</td>
      <td>34,705</td>
      <td>35,669</td>
      <td>36,633</td>
      <td>37,597</td>
      <td>964</td>
    </tr>
    <tr>
      <td>5</td>
      <td>32,357</td>
      <td>33,436</td>
      <td>34,515</td>
      <td>35,594</td>
      <td>36,673</td>
      <td>37,752</td>
      <td>38,831</td>
      <td>39,910</td>
      <td>40,989</td>
      <td>42,068</td>
      <td>1,079</td>
    </tr>
    <tr>
      <td>6</td>
      <td>36,070</td>
      <td>37,272</td>
      <td>38,474</td>
      <td>39,676</td>
      <td>40,878</td>
      <td>42,080</td>
      <td>43,282</td>
      <td>44,484</td>
      <td>45,686</td>
      <td>46,888</td>
      <td>1,202</td>
    </tr>
    <tr>
      <td>7</td>
      <td>40,082</td>
      <td>41,418</td>
      <td>42,754</td>
      <td>44,090</td>
      <td>45,426</td>
      <td>46,762</td>
      <td>48,098</td>
      <td>49,434</td>
      <td>50,770</td>
      <td>52,106</td>
      <td>1,336</td>
    </tr>
    <tr>
      <td>8</td>
      <td>44,389</td>
      <td>45,869</td>
      <td>47,349</td>
      <td>48,829</td>
      <td>50,309</td>
      <td>51,789</td>
      <td>53,269</td>
      <td>54,749</td>
      <td>56,229</td>
      <td>57,709</td>
      <td>1,480</td>
    </tr>
    <tr>
      <td>9</td>
      <td>49,028</td>
      <td>50,662</td>
      <td>52,296</td>
      <td>53,930</td>
      <td>55,564</td>
      <td>57,198</td>
      <td>58,832</td>
      <td>60,466</td>
      <td>62,100</td>
      <td>63,734</td>
      <td>1,634</td>
    </tr>
    <tr>
      <td>10</td>
      <td>53,990</td>
      <td>55,790</td>
      <td>57,590</td>
      <td>59,390</td>
      <td>61,190</td>
      <td>62,990</td>
      <td>64,790</td>
      <td>66,590</td>
      <td>68,390</td>
      <td>70,190</td>
      <td>1,800</td>
    </tr>
    <tr>
      <td>11</td>
      <td>59,319</td>
      <td>61,296</td>
      <td>63,273</td>
      <td>65,250</td>
      <td>67,227</td>
      <td>69,204</td>
      <td>71,181</td>
      <td>73,158</td>
      <td>75,135</td>
      <td>77,112</td>
      <td>1,977</td>
    </tr>
    <tr>
      <td>12</td>
      <td>71,099</td>
      <td>73,469</td>
      <td>75,839</td>
      <td>78,209</td>
      <td>80,579</td>
      <td>82,949</td>
      <td>85,319</td>
      <td>87,689</td>
      <td>90,059</td>
      <td>92,429</td>
      <td>2,370</td>
    </tr>
    <tr>
      <td>13</td>
      <td>84,546</td>
      <td>87,364</td>
      <td>90,182</td>
      <td>93,000</td>
      <td>95,818</td>
      <td>98,636</td>
      <td>101,454</td>
      <td>104,272</td>
      <td>107,090</td>
      <td>109,908</td>
      <td>2,818</td>
    </tr>
    <tr>
      <td>14</td>
      <td>99,908</td>
      <td>103,238</td>
      <td>106,568</td>
      <td>109,898</td>
      <td>113,228</td>
      <td>116,558</td>
      <td>119,888</td>
      <td>123,218</td>
      <td>126,548</td>
      <td>129,878</td>
      <td>3,330</td>
    </tr>
    <tr>
      <td>15</td>
      <td>117,518</td>
      <td>121,435</td>
      <td>125,352</td>
      <td>129,269</td>
      <td>133,186</td>
      <td>137,103</td>
      <td>141,020</td>
      <td>144,937</td>
      <td>148,854</td>
      <td>152,771</td>
      <td>3,917</td>
    </tr>
  </tbody>
</table>
```

## Document Translation

Gemini can translate documents between languages. This example translates meeting notes from English into French and Spanish.

In [ ]:
translation_prompt = """Translate the first paragraph into French and Spanish. Label each paragraph with the target language."""

In [ ]:
# Send Translation Prompt to Gemini
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        translation_prompt,
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/pdf/fdic_board_meeting.pdf",
            mime_type=PDF_MIME_TYPE,
        ),
    ],
    config=GenerateContentConfig(
        temperature=0,
    ),
)

display(Markdown(f"### Translations"))
display(Markdown(response.text))

### Translations

Here are the translations of the first paragraph into French and Spanish:

**French:**

L'activité bancaire a considérablement évolué ces dernières années, et les réglementations doivent évoluer avec le secteur afin de favoriser un système efficace qui réponde aux besoins des entreprises et des consommateurs à travers le pays. En modernisant nos réglementations mettant en œuvre la loi sur le réinvestissement communautaire (CRA), nous espérons promouvoir davantage d'investissements dans les communautés qui en ont le plus besoin.

**Spanish:**

El negocio de la banca ha cambiado drásticamente en los últimos años, y las regulaciones deben evolucionar con la industria para fomentar un sistema eficaz que satisfaga las necesidades de las empresas y los consumidores en todo el país. Al modernizar nuestras regulaciones que implementan la Ley de Reinversión Comunitaria (CRA), esperamos promover mayores inversiones en las comunidades que más las necesitan.


## Document Comparison

Gemini can compare and contrast the contents of multiple documents. This example finds the changes in the IRS Form 1040 between 2013 and 2023.

Note: when working with multiple documents, the order can matter and should be specified in your prompt.

In [ ]:
comparison_prompt = """The first document is from 2013, the second one from 2023. How did the standard deduction evolve?"""

In [ ]:
# Send Comparison Prompt to Gemini
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        comparison_prompt,
        Part.from_uri(
            file_uri=(
                "gs://cloud-samples-data/generative-ai/pdf/form_1040_2013.pdf"
            ),
            mime_type=PDF_MIME_TYPE,
        ),
        Part.from_uri(
            file_uri=(
                "gs://cloud-samples-data/generative-ai/pdf/form_1040_2023.pdf"
            ),
            mime_type=PDF_MIME_TYPE,
        ),
    ],
    config=GenerateContentConfig(temperature=0),
)

display(Markdown(f"### Comparison"))
display(Markdown(response.text))

### Comparison

Here's how the standard deduction evolved between the 2013 and 2023 tax years, based on the provided documents:

**2013 (Form 1040)**

*   **Single or Married Filing Separately:** $6,100
*   **Married Filing Jointly or Qualifying Widow(er):** $12,200
*   **Head of Household:** $8,950

**2023 (Form 1040)**

*   **Single or Married Filing Separately:** $13,850
*   **Married Filing Jointly or Qualifying Surviving Spouse:** $27,700
*   **Head of Household:** $20,800

**Key Observations:**

*   **Significant Increase:** The standard deduction amounts have increased substantially across all filing statuses from 2013 to 2023.
*   **Inflation and Tax Law Changes:** This increase is primarily due to inflation adjustments and changes in tax laws, such as the Tax Cuts and Jobs Act of 2017, which significantly increased the standard deduction.